# Solutions · 02 · Flash separations

Worked solutions to the exercises of [notebook 02](../notebooks/02_flash_separations.ipynb). Try each exercise
yourself before reading its solution - the learning happens in the attempt. Solutions are one way to
answer each question; other correct approaches exist.

In [1]:
try:                      # on Google Colab (or anywhere engsep is missing): install it from GitHub
    import engsep
except ImportError:
    import subprocess, sys
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "git+https://github.com/ktwyw/engsep"], check=True)
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import optimize
from engsep import flash, vle
plt.rcParams.update({"figure.figsize": (7, 4), "figure.dpi": 90, "axes.grid": True, "grid.alpha": 0.3,
                     "axes.spines.top": False, "axes.spines.right": False})

import inspect
from IPython.display import Code

def show_source(obj):
    """Display the source code of a library function or class."""
    return Code(inspect.getsource(obj), language="python")

def heading(text):
    print(f"\n{text}\n" + "=" * len(text))

## Exercise 1

A natural-gas liquid (methane... use propane 0.5, n-butane 0.3, n-pentane 0.2) is flashed at 25 degC. Find the pressure range over which it is two-phase and the vapour fraction at 5 bar. What pressure recovers 90 % of the propane in the vapour?

In [2]:
names, z, T = ["propane", "n-butane", "n-pentane"], [0.5, 0.3, 0.2], 298.15
P_b, P_d = flash.bubble_point_P(names, z, T), flash.dew_point_P(names, z, T)
print(f"at 25 degC the mixture is two-phase between {P_d/1e5:.2f} bar (dew) and {P_b/1e5:.2f} bar (bubble)")
fl = flash.flash_TP(names, z, T, 5e5)
print(f"flash at 5 bar: V/F = {fl['V_over_F']:.3f}, vapour {np.round(fl['y'], 3)}")
P90 = optimize.brentq(lambda P: flash.flash_TP(names, z, T, P)["V_over_F"] * flash.flash_TP(names, z, T, P)["y"][0] / 0.5 - 0.9, P_d * 1.001, P_b * 0.999)
print(f"90 % of the propane in the vapour at {P90/1e5:.2f} bar (V/F = {flash.flash_TP(names, z, T, P90)['V_over_F']:.3f})")

at 25 degC the mixture is two-phase between 2.13 bar (dew) and 5.56 bar (bubble)
flash at 5 bar: V/F = 0.189, vapour [0.806 0.161 0.033]
90 % of the propane in the vapour at 2.96 bar (V/F = 0.739)


Between the dew and bubble pressures the NGL is two-phase; recovering 90 % of the propane overhead needs
most of the feed vaporised, and the vapour is then only mildly enriched - a flash recovers or purifies, not
both, which is the case for a column.

## Exercise 2

Two flashes in series (the liquid from the first flashed again at a lower pressure) versus one flash to the final pressure: compare the purity of the combined vapour for the C4/C5/C6 feed. Why does staging help?

In [3]:
names, z = ["n-butane", "n-pentane", "n-hexane"], [0.3, 0.4, 0.3]
T, P_final = 310.0, 0.85e5
one = flash.flash_TP(names, z, T, P_final)
first = flash.flash_TP(names, z, T, 1.15e5)
second = flash.flash_TP(names, first["x"], T, P_final)
V1, V2 = first["V_over_F"], (1 - first["V_over_F"]) * second["V_over_F"]
y_comb = (V1 * first["y"] + V2 * second["y"]) / (V1 + V2)
print(f"one flash to 0.85 bar: V/F = {one['V_over_F']:.3f}, butane in vapour {one['y'][0]:.3f}")
print(f"two flashes (1.15 bar then 0.85 bar): total V/F = {V1 + V2:.3f}, butane in the combined vapour {y_comb[0]:.3f}")

one flash to 0.85 bar: V/F = 0.774, butane in vapour 0.361
two flashes (1.15 bar then 0.85 bar): total V/F = 0.615, butane in the combined vapour 0.438


The staged flashes produce slightly less vapour but a richer combined product: the first vapour is taken at a
higher pressure, where it is richer in butane, and only the leaner liquid is flashed further. Staging is the
first step towards a column, where the stages are many and the vapours are not combined but refluxed.

## Exercise 3

**Implement it yourself:** write the dew-point temperature equation (sum y_i/K_i = 1) and solve it with brentq for the C4/C5/C6 feed at 2 bar; compare with `flash.dew_point_T`.

In [4]:
T_dew = optimize.brentq(lambda T: sum(zi / vle.k_value(n, T, 2e5) for n, zi in zip(names, z)) - 1, 250, 450)
print(f"by hand: dew point {T_dew - 273.15:.3f} degC; library: {flash.dew_point_T(names, z, 2e5) - 273.15:.3f}")

by hand: dew point 67.389 degC; library: 67.389


The dew-point condition $\sum y_i/K_i = 1$ and a root finder reproduce the library.